# Official CODI GPT-2 validation gate

## Goal

Evaluate the author-released `zen-E/CODI-gpt2` checkpoint before running more mechanistic experiments. The notebook preserves the official architecture, raw-question prompt, six latent iterations, greedy decoding, and benchmark definitions. It never trains or updates weights.

## Setup

### 1. Choose the run scope

The 32-example GSM8K diagnostic checks loading and generation only. It cannot pass the accuracy gate. The full GSM8K run is the primary reproduction gate. Run all four complete benchmarks only after GSM8K passes.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with a pushed immutable commit for the final run.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"

RUN_QUICK_DIAGNOSTIC = True
RUN_FULL_GSM8K_GATE = True
RUN_ALL_FULL_BENCHMARKS = False
QUICK_LIMIT = 32

assert QUICK_LIMIT > 0

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pathlib
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Hugging Face authentication loaded")
except Exception:
    print("Using public Hugging Face access")

if not pathlib.Path(REPO_DIR, ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-official-codi.txt"],
    check=True,
)
os.chdir(REPO_DIR)
resolved_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", resolved_commit)
if RUN_COMMIT == "main":
    print("For the final run, pin RUN_COMMIT to:", resolved_commit)

### 2. Verify the environment and contracts

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a Colab GPU runtime"
print("Torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_official_codi.py", "tests/test_datasets.py"],
    cwd=REPO_DIR,
    check=True,
)

## Steps

### 3. Define Drive-persistent execution

Predictions, manifests, summaries, and logs are written directly to Drive. The downloaded model remains in the runtime Hugging Face cache and can be downloaded again if the runtime ends.

In [ ]:
import datetime
import json

OUTPUT_ROOT = pathlib.Path(DRIVE_ROOT) / "outputs" / "official_codi_gpt2"
LOG_ROOT = pathlib.Path(DRIVE_ROOT) / "logs" / "official_codi_gpt2"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def run_persisted(arguments, log_name):
    command = [sys.executable, "-u", "-m", "src.eval.official_codi", *arguments]
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(command), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} ===\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"official CODI evaluation failed with exit code {return_code}")
    return log_path

### 4. Run the loading and generation diagnostic

In [ ]:
if RUN_QUICK_DIAGNOSTIC:
    run_persisted(
        [
            "--config", "configs/official_codi_gpt2.yaml",
            "--datasets", "gsm8k",
            "--limit", str(QUICK_LIMIT),
            "--device", "cuda",
            "--output-dir", str(OUTPUT_ROOT),
        ],
        f"gsm8k_limit{QUICK_LIMIT}.log",
    )
else:
    print("Quick diagnostic skipped")

### 5. Run the complete GSM8K accuracy gate

In [ ]:
if RUN_FULL_GSM8K_GATE:
    run_persisted(
        [
            "--config", "configs/official_codi_gpt2.yaml",
            "--datasets", "gsm8k",
            "--limit", "0",
            "--device", "cuda",
            "--output-dir", str(OUTPUT_ROOT),
        ],
        "gsm8k_full.log",
    )
else:
    print("Full GSM8K gate skipped")

### 6. Optionally run every complete official benchmark

In [ ]:
if RUN_ALL_FULL_BENCHMARKS:
    primary_summary_path = next(OUTPUT_ROOT.glob("eval/revision_*/full_gsm8k/summary.json"), None)
    assert primary_summary_path is not None, "Run the full GSM8K gate first"
    primary_summary = json.loads(primary_summary_path.read_text())
    assert primary_summary["accuracy_gate"]["status"] == "passed", (
        "Do not continue to all benchmarks until the GSM8K compatibility gate passes"
    )
    run_persisted(
        [
            "--config", "configs/official_codi_gpt2.yaml",
            "--limit", "0",
            "--device", "cuda",
            "--output-dir", str(OUTPUT_ROOT),
        ],
        "all_benchmarks_full.log",
    )
else:
    print("All-benchmark evaluation skipped. Enable only after GSM8K passes.")

## Checks

### 7. Inspect the durable summaries

In [ ]:
summaries = sorted(OUTPUT_ROOT.glob("eval/revision_*/*/summary.json"))
assert summaries, "No official CODI summaries were produced"
for summary_path in summaries:
    summary = json.loads(summary_path.read_text())
    print("\n", summary_path)
    print(json.dumps({
        "official_last_number_accuracy": summary["datasets"],
        "numeric_exact_match_accuracy": summary["numeric_exact_match_datasets"],
        "evaluated_counts": summary["evaluated_counts"],
        "gate": summary["accuracy_gate"]["status"],
    }, indent=2))

## Next Steps

- If full GSM8K passes, run the complete four-benchmark evaluation.
- Then implement official-checkpoint KV extraction and recompute Stage 1b and Stage 1c. Do not reuse the projector learned from the pilot checkpoint.
- If GSM8K fails, stop before spectral analysis and compare generated text, checkpoint load coverage, package versions, and official repository inference behavior.